# NeuroBot — export of the finger-level models (S01, TCFormer, ONNX)

Trains one TCFormer per configuration (**2 fingers: thumb/pinky; 3 fingers: thumb/index/pinky**) on subject **S01** of the base paper's dataset, with the same session-held-out protocol as the finger notebook v2.1, and exports the fine-tuned **ME** and **MI** models to ONNX for the web app. Seed 0, no cross-subject pretraining, TCFormer only (the EEGNet arm is not needed for deployment).

**Before the first full run:** keep `DRY_RUN = True` once (3-epoch smoke test, needs the same 16 GB download), check that the four ONNX parity lines print, then set `DRY_RUN = False`. Enable GPU and Internet. The training time is an estimate and has not been measured: read the `trained in ... min` line.

**Output:** `neurobot_finger_models.zip` (4 `.onnx` files, `manifest_entries.json`, replay logs without waveform) and `finger_results_export.json`. Download the zip from the Output tab and send it back.

The backend does not serve 128-channel finger models yet; that needs a small change that is only worth making once these files exist.


In [ ]:
import os, re, glob, json, copy, time, shutil, zipfile, tarfile, warnings
warnings.filterwarnings("ignore")
import numpy as np
import scipy.io, scipy.signal
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ------------------------------ configuration ------------------------------
DRY_RUN       = True            # True: 1 subject, 2- AND 3-class, both models, 1 seed, 3 epochs (writes ..._DRYRUN.json). False = the real run.
SUBJECTS      = [1]             # 1..21; one subject per Kaggle version (each is a 16-24 GB download plus hours of training)
NCLASS_LIST   = [2, 3]          # 4 is not supported (label 3 exists only in the offline session)
MODES         = ["ME", "MI"]
DATA_ROOT     = None            # or e.g. "/kaggle/input/finger-bci" if you attached data containing S01/, S02/, ...
FIGSHARE_ID   = 29104040
SUBJECT_FILE_MAP = {}           # optional {subject_id: ["exact_file_name.zip", ...]} if auto-matching fails

TMP_DIR   = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
WORK_DIR  = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
CACHE_DIR = os.path.join(WORK_DIR, "finger_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

PARAMS = dict(maxtriallen=5, windowlen=1.0, step_sec=0.125, downsrate=100, bandpass=(4, 40))
ZERO_PHASE = False              # official code uses a causal lfilter per window; True = filtfilt
FT_FRAC = 0.30                  # share of the held-out session's first trials used for fine-tuning
TWO_CLASS_FOLDERS = "all"       # "all" (as the official code) or "2class" (only the 2class online folders)
MODELS_TO_RUN = ["tcformer"]       # EXPORT notebook: TCFormer only
SEEDS = {"tcformer": [0], "eegnet": [0]}   # EXPORT notebook: one seed (seed 0)
EXT_SUBJECTS = []               # optional: e.g. [2, 3] -> offline windows of those subjects used for cross-subject pretraining
EXT_STRIDE = 4                  # keep every 4th window of external subjects (memory)
EXT_EPOCHS, EXT_PATIENCE = 30, 5
EPOCH_OVERRIDE = None
MAX_RUNTIME_HOURS = 10.5        # Kaggle kills a session at 12 h and then keeps NO output; stop starting new work before that
MIN_BLOCK_MINUTES = 60          # do not start a new class configuration / model block with less than this left
PRIOR_RESULTS_PATHS = []        # e.g. ["/kaggle/input/my-finger-run-1/finger_results_export.json"] to continue an earlier version
T0 = time.time()
if DRY_RUN:
    SUBJECTS, NCLASS_LIST = SUBJECTS[:1], [2, 3]
    SEEDS = {"tcformer": [0], "eegnet": [0]}
    EXT_SUBJECTS, EXT_EPOCHS, EXT_PATIENCE, EPOCH_OVERRIDE = [], 2, 2, 3
VARIANTS = ["baseline"] + (["cross"] if EXT_SUBJECTS else [])
CLASS_SETS = {2: [1, 4], 3: [1, 2, 4]}      # label ids used by the official code: 1 thumb, 2 index, 4 pinky
assert set(NCLASS_LIST) <= {2, 3}, "only 2- and 3-class are supported (label 3 exists only in the offline session)"
assert not set(EXT_SUBJECTS) & set(SUBJECTS), "external subjects must never include evaluated subjects"
RESULTS_PATH = os.path.join(WORK_DIR, "finger_results_v2_DRYRUN.json" if DRY_RUN else "finger_results_export.json")

def time_left():
    return MAX_RUNTIME_HOURS * 3600 - (time.time() - T0)

def _json_default(o):
    if hasattr(o, "item"): return o.item()          # numpy scalars
    if hasattr(o, "tolist"): return o.tolist()      # numpy arrays
    return str(o)

def load_json_safe(path):
    """{} for a missing, empty (0-byte) or corrupt file instead of crashing."""
    try:
        if os.path.exists(path) and os.path.getsize(path) > 0:
            with open(path) as fh:
                return json.load(fh)
    except Exception as e:
        print(f"  could not read {path} ({type(e).__name__}: {e}) - ignoring it")
    return {}

def load_results_merged():
    merged = {}
    for p in list(PRIOR_RESULTS_PATHS) + [RESULTS_PATH]:
        part = load_json_safe(p)
        if part: print(f"  loaded {len(part)} record(s) from {p}")
        merged.update(part)
    return merged

def save_results(res):
    """Atomic: write a temp file, flush to disk, then rename over the real file (never leaves a half-written/empty JSON)."""
    tmp = RESULTS_PATH + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(res, fh, indent=1, default=_json_default)
        fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, RESULTS_PATH)
print(f"results file: {RESULTS_PATH}")
print(f"mode={'DRY_RUN' if DRY_RUN else 'FULL'} | subjects={SUBJECTS} | classes={NCLASS_LIST} | models={MODELS_TO_RUN} | "
      f"variants={VARIANTS} | seeds={SEEDS}")

## 1. Training helpers (same as the limb notebooks) + EEGNet

In [ ]:
import random

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_loader(X, y, batch_size=16, shuffle=True):
    ds = torch.utils.data.TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def class_weights(y_int, n_classes):
    counts = np.bincount(y_int, minlength=n_classes).astype(np.float32)
    weights = counts.sum() / (n_classes * np.maximum(counts, 1))
    return torch.tensor(weights, dtype=torch.float32)

def augment_batch(xb, noise_std=0.05, max_shift=8):
    noise = torch.randn_like(xb) * noise_std
    xb = xb + noise
    shift = np.random.randint(-max_shift, max_shift + 1)
    if shift != 0:
        xb = torch.roll(xb, shifts=shift, dims=-1)
    return xb


class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, base_lr):
        self.optimizer = optimizer
        self.warmup_steps = max(1, warmup_steps)
        self.total_steps = max(total_steps, self.warmup_steps + 1)
        self.base_lr = base_lr
        self.step_count = 0

    def step_batch(self):
        self.step_count += 1
        if self.step_count <= self.warmup_steps:
            lr = self.base_lr * self.step_count / self.warmup_steps
        else:
            progress = (self.step_count - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            progress = min(progress, 1.0)
            lr = 0.5 * self.base_lr * (1 + np.cos(np.pi * progress))
        for g in self.optimizer.param_groups:
            g["lr"] = lr

    def step_epoch(self):
        pass


def train_loop(model, loader, val_loader, optimizer, criterion, max_epochs, patience=20,
                verbose=True, augment=True, scheduler=None):
    own_scheduler = scheduler is None
    if own_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8)
    best_val_loss = float('inf')
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    for epoch in range(max_epochs):
        model.train()
        total_loss, total_correct, total_n = 0.0, 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if augment:
                xb = augment_batch(xb)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward(); optimizer.step()
            if scheduler is not None and hasattr(scheduler, "step_batch"):
                scheduler.step_batch()
            total_loss += loss.item() * xb.size(0)
            total_correct += (logits.argmax(1) == yb).sum().item()
            total_n += xb.size(0)
        train_loss = total_loss / total_n
        train_acc = total_correct / total_n

        model.eval()
        val_loss, val_correct, val_n = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                val_loss += criterion(logits, yb).item() * xb.size(0)
                val_correct += (logits.argmax(1) == yb).sum().item()
                val_n += xb.size(0)
        val_loss /= val_n
        val_acc = val_correct / val_n

        if own_scheduler:
            scheduler.step(val_loss)
        elif hasattr(scheduler, "step_epoch"):
            scheduler.step_epoch()

        history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss,
                         "train_acc": train_acc, "val_acc": val_acc,
                         "lr": optimizer.param_groups[0]["lr"]})

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(f"    epoch {epoch+1}/{max_epochs}  train_loss={train_loss:.4f} train_acc={train_acc:.3f}  "
                  f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}  lr={optimizer.param_groups[0]['lr']:.1e}")

        if epochs_without_improvement >= patience:
            if verbose:
                print(f"    early stopping at epoch {epoch+1} (no val improvement for {patience} epochs); "
                      f"best epoch was {best_epoch} (val_loss={best_val_loss:.4f})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


@torch.no_grad()
def evaluate(model, loader, label_order):
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        p = model(xb.to(DEVICE)).argmax(1).cpu().numpy()
        preds.extend(p); trues.extend(yb.numpy())
    acc = accuracy_score(trues, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(trues, preds, average=None,
                                                          labels=range(len(label_order)), zero_division=0)
    cm = confusion_matrix(trues, preds, labels=range(len(label_order)))
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "confusion_matrix": cm}

@torch.no_grad()
def evaluate_majority_vote(model, X, y_int, trial_ids, label_order):
    model.eval()
    loader = make_loader(X, y_int, shuffle=False)
    window_preds = []
    for xb, _ in loader:
        window_preds.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
    window_preds = np.array(window_preds)

    trial_true, trial_pred = [], []
    for t in np.unique(trial_ids):
        mask = trial_ids == t
        trial_true.append(y_int[mask][0])
        trial_pred.append(np.bincount(window_preds[mask], minlength=len(label_order)).argmax())
    acc = accuracy_score(trial_true, trial_pred)
    cm = confusion_matrix(trial_true, trial_pred, labels=range(len(label_order)))
    return {"accuracy": acc, "n_trials": len(trial_true), "confusion_matrix": cm}


def plot_training_curve(history, title):
    epochs = [h["epoch"] for h in history]
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    axes[0].plot(epochs, [h["train_loss"] for h in history], label="train")
    axes[0].plot(epochs, [h["val_loss"] for h in history], label="val")
    axes[0].set_title(f"{title} - loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(epochs, [h["train_acc"] for h in history], label="train")
    axes[1].plot(epochs, [h["val_acc"] for h in history], label="val")
    axes[1].set_ylim(0, 1); axes[1].set_title(f"{title} - accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
    axes[2].plot(epochs, [h["lr"] for h in history], color="#888888")
    axes[2].set_title(f"{title} - learning rate"); axes[2].set_xlabel("epoch")
    plt.tight_layout(); plt.show()
    best_epoch = max(range(len(history)), key=lambda i: history[i]["val_acc"])
    print(f"  Ran {len(history)} epochs. Best val accuracy {history[best_epoch]['val_acc']:.3f} "
          f"at epoch {history[best_epoch]['epoch']} (final epoch val_acc={history[-1]['val_acc']:.3f}).")

def freeze_early_layers(model, keep_frac=0.3):
    params = list(model.named_parameters())
    cutoff = int(len(params) * (1 - keep_frac))
    for i, (name, p) in enumerate(params):
        p.requires_grad = (i >= cutoff)
    return [p for p in model.parameters() if p.requires_grad]


def make_optimizer(params, name, lr, weight_decay):
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)




def split_by_trial(trial_ids, y_int, test_size, seed=0):
    """Boolean masks (train_mask, test_mask) such that every trial goes wholly to one side."""
    uniq = np.unique(trial_ids)
    lab = np.array([y_int[trial_ids == t][0] for t in uniq])
    try:
        tr, te = train_test_split(uniq, test_size=test_size, stratify=lab, random_state=seed)
    except ValueError:
        tr, te = train_test_split(uniq, test_size=test_size, random_state=seed)
    return np.isin(trial_ids, tr), np.isin(trial_ids, te)

class EEGNet(nn.Module):
    def __init__(self, n_channels, n_samples, n_classes, F1=8, D=2, F2=16, kernel_length=32, dropout=0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length // 2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D), nn.ELU(),
            nn.AvgPool2d((1, 4)), nn.Dropout(dropout),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2), nn.ELU(),
            nn.AvgPool2d((1, 8)), nn.Dropout(dropout),
        )
        self.eval()                               # size probe in eval mode: must not touch BatchNorm statistics
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            flat_dim = self.block2(self.block1(dummy)).numel()
        self.train()
        self.classifier = nn.Linear(flat_dim, n_classes)

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(1)
        x = self.block1(x); x = self.block2(x)
        return self.classifier(x.flatten(1))

    def set_dropout(self, p):
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.p = p

def eegnet_factory(n_channels, n_samples, n_classes):
    return EEGNet(n_channels, n_samples, n_classes, dropout=0.5)

## 2. Getting one subject's raw data (Figshare API) with a disk check

In [ ]:
import requests

def list_figshare_files(article_id=FIGSHARE_ID, show=True):
    files, page = [], 1
    while True:
        r = requests.get(f"https://api.figshare.com/v2/articles/{article_id}/files",
                         params={"page": page, "page_size": 100}, timeout=60)
        r.raise_for_status()
        chunk = r.json()
        if not chunk:
            break
        files += chunk; page += 1
    if show:
        total = sum(f["size"] for f in files)
        print(f"{len(files)} files, {total / 1e9:.1f} GB total")
        for f in sorted(files, key=lambda f: f["name"]):
            print(f"  {f['size'] / 1e9:8.2f} GB  {f['name']}")
    return files

def files_for_subject(files, subj):
    if subj in SUBJECT_FILE_MAP:
        wanted = set(SUBJECT_FILE_MAP[subj]); return [f for f in files if f["name"] in wanted]
    pat = re.compile(rf"(^|[^A-Za-z0-9])S(ubject)?[_ -]?0*{subj}([^0-9]|$)", re.I)
    return [f for f in files if pat.search(f["name"])]

def download(url, dest, expected_size=None):
    done, t0 = 0, time.time()
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest, "wb") as fh:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                fh.write(chunk); done += len(chunk)
                if int((time.time() - t0)) % 60 == 0 and done % (64 * 1024 * 1024) < 8 * 1024 * 1024:
                    print(f"    {done / 1e9:.2f} GB downloaded", end="\r")
    if expected_size and done != expected_size:
        raise IOError(f"incomplete download: {done} of {expected_size} bytes")

def fetch_subject_raw(subj):
    """Returns (folder_with_raw_data, cleanup_fn)."""
    if DATA_ROOT:
        return DATA_ROOT, (lambda: None)
    files = files_for_subject(list_figshare_files(show=False), subj)
    if not files:
        raise FileNotFoundError(f"No Figshare file matched subject {subj}. Run list_figshare_files() and fill SUBJECT_FILE_MAP.")
    need = sum(f["size"] for f in files) * 2.2          # archive + extracted copy
    free = shutil.disk_usage(TMP_DIR).free
    if free < need:
        raise OSError(f"Subject {subj} needs ~{need / 1e9:.0f} GB scratch but only {free / 1e9:.0f} GB is free in {TMP_DIR}. "
                      "Use a larger-disk environment, or attach pre-extracted data and set DATA_ROOT.")
    out = os.path.join(TMP_DIR, f"finger_S{subj:02d}")
    shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
    for f in files:
        dest = os.path.join(out, f["name"])
        print(f"  downloading {f['name']} ({f['size'] / 1e9:.2f} GB)")
        for attempt in range(3):
            try:
                download(f["download_url"], dest, f["size"]); break
            except Exception as e:
                print(f"    download attempt {attempt + 1}/3 failed ({type(e).__name__}: {e})")
                if os.path.exists(dest): os.remove(dest)
                if attempt == 2: raise
                time.sleep(30 * (attempt + 1))
        if zipfile.is_zipfile(dest):
            with zipfile.ZipFile(dest) as z: z.extractall(out)
            os.remove(dest)
        elif tarfile.is_tarfile(dest):
            with tarfile.open(dest) as t: t.extractall(out)
            os.remove(dest)
    return out, (lambda: shutil.rmtree(out, ignore_errors=True))

## 3. Loader and preprocessing (mirrors the official code)

`cache_subject` now also stores the **folder name** of every window (needed to separate `2class` from `3class` trials),
and can build a lightweight offline-only, strided cache for external subjects. Caches are named `S<nn>_v21*.npz`, so v1/v2
caches are not reused (v1 lacks the folder information; v2 sorted file names as text).

In [ ]:
def load_mat_trials(path, maxtriallen=5):
    """-> trials (n, n_chan, maxtriallen*srate) NaN-padded, labels (n,), srate. CAR is applied here, as in the official code."""
    mat = scipy.io.loadmat(path)
    eeg, event = mat["eeg"], mat["event"]
    signals = eeg["data"][0][0]
    srate = float(np.squeeze(eeg["fsample"][0][0]))
    starts, ends, targets = [], [], []
    for i in range(event.shape[1]):
        evt = event[0, i]
        etype = str(np.squeeze(evt["type"]))
        sample = int(np.squeeze(evt["sample"]))
        value = np.squeeze(evt["value"])
        if etype == "Target":
            starts.append(sample - 1); targets.append(int(value))
        elif etype == "TrialEnd":
            ends.append(sample - 1)
    n_max = int(maxtriallen * srate)
    n = min(len(starts), len(ends))
    trials = np.full((n, signals.shape[0], n_max), np.nan, dtype=np.float32)
    for k in range(n):
        seg = signals[:, starts[k]:ends[k]][:, :n_max]
        trials[k, :, :seg.shape[1]] = seg
    trials = trials - trials.mean(axis=1, keepdims=True)          # common average reference
    return trials, np.array(targets[:n]), srate


def preprocess_trials(trials, srate, p=PARAMS, zero_phase=ZERO_PHASE):
    """-> windows (m, n_chan, 100) float32, trial_index (m,). Windows touching the NaN padding are dropped."""
    seg = int(p["windowlen"] * srate); step = int(p["step_sec"] * srate)
    desired = int(p["windowlen"] * p["downsrate"])
    b, a = scipy.signal.butter(4, list(p["bandpass"]), btype="bandpass", fs=p["downsrate"])
    W, T = [], []
    for ti in range(trials.shape[0]):
        tr = trials[ti]
        starts = [s for s in range(0, tr.shape[1] - seg + 1, step) if not np.isnan(tr[:, s:s + seg]).any()]
        if not starts:
            continue
        x = np.stack([tr[:, s:s + seg] for s in starts])                    # (k, chan, seg)
        x = scipy.signal.resample(x, desired, axis=2)                        # 1024 Hz -> 100 Hz
        if zero_phase:
            x = scipy.signal.filtfilt(b, a, x, axis=-1)
        else:
            pad = 100
            xp = np.pad(x, ((0, 0), (0, 0), (pad, pad)))
            x = scipy.signal.lfilter(b, a, xp, axis=-1)[:, :, pad:-pad]
        x = (x - x.mean(axis=2, keepdims=True)) / (x.std(axis=2, keepdims=True) + 1e-8)   # z-score per channel
        W.append(x.astype(np.float32)); T.extend([ti] * len(x))
    if not W:
        return np.zeros((0, trials.shape[1], desired), np.float32), np.zeros(0, int)
    return np.concatenate(W), np.array(T)


def describe_folder(folder):
    """mode / session / class-setting parsed from a data-folder name (names follow the official glob patterns)."""
    low = folder.lower()
    mode = "ME" if "movement" in low else "MI" if "imagery" in low else None
    m = re.search(r"sess(\d+)", low)
    return mode, (int(m.group(1)) if m else 0), ("3class" if "3class" in low else "2class" if "2class" in low else "")


def _natkey(s):
    """Natural sort key: 'Run2' < 'Run10' (plain text sorting would put Run10 first and scramble the chronology)."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(s))]


def discover_mat_files(root, subj):
    base = os.path.join(root, f"S{subj:02d}")
    if not os.path.isdir(base):
        cands = [d for d, _, _ in os.walk(root) if os.path.basename(d).upper() == f"S{subj:02d}"]
        base = cands[0] if cands else root
    found = []
    for dirpath, _, fnames in os.walk(base):
        mats = sorted((f for f in fnames if f.lower().endswith(".mat")), key=_natkey)
        if not mats:
            continue
        mode, sess, cls = describe_folder(os.path.basename(dirpath))
        if mode is None:
            continue
        for f in mats:
            found.append(dict(path=os.path.join(dirpath, f), mode=mode, session=sess, folder=os.path.basename(dirpath)))
    found.sort(key=lambda d: (d["mode"], d["session"], _natkey(d["folder"]), _natkey(os.path.basename(d["path"]))))
    if not found:
        raise FileNotFoundError(f"No .mat files under folders named *Movement*/*Imagery* in {base}. Check the extracted layout.")
    return found


def cache_subject(subj, force=False, offline_only=False, stride=1):
    tag = f"_offline_s{stride}" if offline_only else ""
    path = os.path.join(CACHE_DIR, f"S{subj:02d}_v21{tag}.npz")
    if os.path.exists(path) and not force:
        return path
    root, cleanup = fetch_subject_raw(subj)
    try:
        X, Y, TID, MODE, SESS, FOLD = [], [], [], [], [], []
        uid = 0
        for d in discover_mat_files(root, subj):
            if offline_only and d["session"] != 0:
                continue
            trials, labels, srate = load_mat_trials(d["path"], PARAMS["maxtriallen"])
            w, ti = preprocess_trials(trials, srate)
            if len(w) == 0:
                continue
            keep = np.arange(0, len(w), stride)
            w, ti = w[keep], ti[keep]
            X.append(w.astype(np.float16)); Y.append(labels[ti]); TID.append(ti + uid)
            MODE.append(np.full(len(w), d["mode"])); SESS.append(np.full(len(w), d["session"]))
            FOLD.append(np.full(len(w), d["folder"]))
            uid += trials.shape[0]
            print(f"  S{subj:02d} {d['mode']} sess{d['session']} {d['folder']}/{os.path.basename(d['path'])}: {trials.shape[0]} trials, {len(w)} windows")
        np.savez_compressed(path, X=np.concatenate(X), y=np.concatenate(Y), trial=np.concatenate(TID),
                            mode=np.concatenate(MODE), session=np.concatenate(SESS), folder=np.concatenate(FOLD))
    finally:
        cleanup()
    return path

def load_cache(path):
    with np.load(path) as f:
        return {k: f[k] for k in f.files}

## 4. Audit: what is actually in the data?

Caches the first subject, then prints trial counts per label for every mode and session. Read it before training.

In [ ]:
def audit_subject(subj):
    z = load_cache(cache_subject(subj))
    print(f"\nSubject {subj}: {len(z['y'])} windows total")
    for mode in MODES:
        for sess in sorted(set(z["session"][z["mode"] == mode].tolist())):
            m = (z["mode"] == mode) & (z["session"] == sess)
            trials_per = {int(l): int(len(np.unique(z["trial"][m & (z["y"] == l)]))) for l in np.unique(z["y"][m])}
            folders = sorted(set(z["folder"][m].tolist()))
            print(f"  {mode} session {sess}: trials per label {trials_per} | {len(folders)} folder(s)")
    return z

_z = audit_subject(SUBJECTS[0])
labels_present = set(np.unique(_z["y"]).tolist())
print("\nlabels present:", sorted(labels_present))
FEASIBLE = [k for k in NCLASS_LIST if set(CLASS_SETS[k]) <= labels_present]
for k in NCLASS_LIST:
    if k not in FEASIBLE:
        print(f"-> {k}-class SKIPPED: labels {sorted(set(CLASS_SETS[k]) - labels_present)} not found.")
print("configurations that will run:", FEASIBLE)
del _z

## 5. Models for 128 channels x 100 samples: TCFormer (pooling chosen automatically) and EEGNet

In [ ]:
!pip install -q braindecode
from braindecode.models import TCFormer

N_CHANS, N_TIMES = 128, int(PARAMS["windowlen"] * PARAMS["downsrate"])
POOL_CANDIDATES = [(4, 2), (2, 2), (2, 1)]
CHOSEN_POOL = None
for p1, p2 in POOL_CANDIDATES:
    try:
        m = TCFormer(n_chans=N_CHANS, n_times=N_TIMES, n_outputs=3, pool_length_1=p1, pool_length_2=p2)
        tok = {}
        m.to_tokens.register_forward_hook(lambda mod, inp, out: tok.update(n=out.shape[1]))
        m(torch.randn(2, N_CHANS, N_TIMES))
        CHOSEN_POOL = (p1, p2)
        print(f"TCFormer pooling {CHOSEN_POOL}: {sum(q.numel() for q in m.parameters())} params, {tok['n']} transformer tokens")
        break
    except Exception as e:
        print(f"pooling {(p1, p2)} failed: {type(e).__name__}: {e}")
assert CHOSEN_POOL, "no TCFormer pooling configuration worked for 128 x 100 inputs"

def tcformer_factory(n_channels, n_samples, n_classes):
    return TCFormer(n_chans=n_channels, n_times=n_samples, n_outputs=n_classes,
                    pool_length_1=CHOSEN_POOL[0], pool_length_2=CHOSEN_POOL[1])

KW = dict(optimizer_name="adamw", weight_decay=1e-2, label_smoothing=0.1, keep_frac=0.5,
          use_warmup=True, warmup_frac=0.1, base_lr=5e-4, ft_lr=3e-4)

In [ ]:
MODEL_CFG = {
    "tcformer": dict(factory=tcformer_factory,
                     kw=dict(optimizer_name="adamw", weight_decay=1e-2, label_smoothing=0.1, keep_frac=0.5,
                             use_warmup=True, warmup_frac=0.1, base_lr=5e-4, ft_lr=3e-4),
                     cross_base_lr=3e-4, base_epochs=150, ft_epochs=80, base_patience=20, ft_patience=15),
    "eegnet": dict(factory=eegnet_factory,
                   kw=dict(optimizer_name="adam", weight_decay=0.0, label_smoothing=0.0, keep_frac=0.3,
                           use_warmup=False, warmup_frac=0.1, base_lr=1e-3, ft_lr=5e-4),
                   cross_base_lr=5e-4, base_epochs=150, ft_epochs=60, base_patience=20, ft_patience=12),
}
if EPOCH_OVERRIDE:
    for _m in MODEL_CFG.values():
        _m.update(base_epochs=EPOCH_OVERRIDE, ft_epochs=EPOCH_OVERRIDE, base_patience=EPOCH_OVERRIDE, ft_patience=EPOCH_OVERRIDE)

@torch.no_grad()
def predict_proba(model, X, batch=128):
    model.eval(); out = []
    for i in range(0, len(X), batch):
        xb = torch.tensor(X[i:i + batch], dtype=torch.float32).to(DEVICE)
        out.append(torch.softmax(model(xb), dim=1).cpu().numpy())
    return np.concatenate(out)

def trial_vote_accuracy(pred, y, trial_ids, n_classes):
    true_l, pred_l = [], []
    for t in np.unique(trial_ids):
        m = trial_ids == t
        true_l.append(y[m][0]); pred_l.append(np.bincount(pred[m], minlength=n_classes).argmax())
    return float(np.mean(np.array(true_l) == np.array(pred_l))), len(true_l)

_e = eegnet_factory(N_CHANS, N_TIMES, 3)
print("EEGNet sanity check:", sum(p.numel() for p in _e.parameters()), "params, output", tuple(_e(torch.randn(2, N_CHANS, N_TIMES)).shape))

## 6. Plan, verify, train

`plan_finger` builds the chronological, trial-level plan for one class configuration:

* 2-class: labels 1 and 4 from all online folders (as the official code) — or only `2class` folders with `TWO_CLASS_FOLDERS = "2class"`.
* 3-class: labels 1, 2, 4 from the **`3class` online folders only** (plus the offline session).
* test session = the last session of each mode; pretraining uses all earlier sessions of both modes;
  fine-tune = first `FT_FRAC` of the test session's trials; evaluation = the rest.

`verify_finger_plan` raises an error on any trial overlap, on non-chronological evaluation, or on a class missing from
the training data.

In [ ]:
def plan_finger(z, nclass):
    keep = CLASS_SETS[nclass]; lab_to_int = {l: i for i, l in enumerate(keep)}
    sess_all, fol_all = z["session"], z["folder"].astype(str)
    sel = np.isin(z["y"], keep)
    if nclass == 3:
        sel &= (sess_all == 0) | (np.char.find(fol_all, "3class") >= 0)
    elif TWO_CLASS_FOLDERS == "2class":
        sel &= (sess_all == 0) | (np.char.find(fol_all, "2class") >= 0)
    idx = np.where(sel)[0]
    X = z["X"][idx]; trial = z["trial"][idx]; mode = z["mode"][idx]; sess = sess_all[idx]
    y = np.array([lab_to_int[int(l)] for l in z["y"][idx]])
    test_sess = {m: int(sess[mode == m].max()) for m in MODES
                 if (mode == m).any() and len(set(sess[mode == m].tolist())) >= 2}
    assert test_sess, "no mode has an offline session plus at least one later session"
    base_mask = np.zeros(len(y), bool)
    for m, ts in test_sess.items():
        base_mask |= (mode == m) & (sess < ts)
    idx_base = np.where(base_mask)[0]
    tr_m, va_m = split_by_trial(trial[idx_base], y[idx_base], 0.15)
    plan = dict(X=X, y=y, trial=trial, mode=mode, sess=sess, nclass=nclass, label_order=[f"label{l}" for l in keep],
                base_train=idx_base[tr_m], base_val=idx_base[va_m], test_sess=test_sess, modes={})
    for m, ts in test_sess.items():
        idm = np.where((mode == m) & (sess == ts))[0]
        order = np.unique(trial[idm])
        n_ft = max(1, int(round(len(order) * FT_FRAC)))
        ft_idx = idm[np.isin(trial[idm], order[:n_ft])]
        ev_idx = idm[np.isin(trial[idm], order[n_ft:])]
        if len(np.unique(trial[ev_idx])) < 3:
            print(f"  {m}: skipped (fewer than 3 evaluation trials)"); continue
        a, b = split_by_trial(trial[ft_idx], y[ft_idx], 0.2)
        ft_train, ft_val = ft_idx[a], ft_idx[b]
        if len(ft_val) == 0:
            ft_val = ft_train
        if len(np.unique(y[ft_train])) < nclass:
            print(f"  {m}: skipped (a class is missing from the fine-tune trials)"); continue
        plan["modes"][m] = dict(ft_train=ft_train, ft_val=ft_val, eval=ev_idx)
    return plan

def verify_finger_plan(plan, label):
    T = lambda idx: set(plan["trial"][idx].tolist())
    nclass = plan["nclass"]
    assert len(plan["modes"]) > 0, f"{label}: no mode can be evaluated"
    assert len(np.unique(plan["y"][plan["base_train"]])) == nclass, f"{label}: a class is missing from the pretraining data"
    assert not (T(plan["base_train"]) & T(plan["base_val"])), f"{label}: pretrain and its validation share a trial"
    for m, p in plan["modes"].items():
        parts = {"pretrain": T(plan["base_train"]), "pretrain_val": T(plan["base_val"]),
                 "ft_train": T(p["ft_train"]), "ft_val": T(p["ft_val"]), "eval": T(p["eval"])}
        names = list(parts)
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                if names[i] == "ft_train" and names[j] == "ft_val" and p["ft_val"] is p["ft_train"]:
                    continue
                shared = parts[names[i]] & parts[names[j]]
                assert not shared, f"{label} {m}: LEAKAGE - {len(shared)} trial(s) in both {names[i]} and {names[j]}"
        ts = plan["test_sess"][m]
        assert (plan["sess"][p["eval"]] == ts).all() and (plan["sess"][p["ft_train"]] == ts).all(), f"{label} {m}: wrong session"
        assert (plan["sess"][plan["base_train"]][plan["mode"][plan["base_train"]] == m] < ts).all(), f"{label} {m}: pretraining includes the test session"
        assert plan["trial"][p["ft_train"]].max() < plan["trial"][p["eval"]].min(), f"{label} {m}: evaluation is not strictly later than fine-tuning"
        assert (plan["mode"][p["eval"]] == m).all(), f"{label} {m}: evaluation windows from another mode"
    return {m: len(T(p["eval"])) for m, p in plan["modes"].items()}

KEEP_MODELS = True      # EXPORT notebook: keep each fine-tuned model so it can be written to ONNX

def run_finger_once(plan, model_name, variant, seed, ext_state=None):
    mc = MODEL_CFG[model_name]; kw = dict(mc["kw"]); nclass = plan["nclass"]
    set_seed(seed)
    base = mc["factory"](N_CHANS, N_TIMES, nclass).to(DEVICE)
    if variant == "cross":
        base.load_state_dict(ext_state); kw["base_lr"] = mc["cross_base_lr"]
    X, y = plan["X"], plan["y"]
    Xb, yb = X[plan["base_train"]].astype(np.float32), y[plan["base_train"]]
    Xv, yv = X[plan["base_val"]].astype(np.float32), y[plan["base_val"]]
    crit = nn.CrossEntropyLoss(weight=class_weights(yb, nclass).to(DEVICE), label_smoothing=kw["label_smoothing"])
    opt = make_optimizer(base.parameters(), kw["optimizer_name"], lr=kw["base_lr"], weight_decay=kw["weight_decay"])
    loader = make_loader(Xb, yb)
    sched = None
    if kw["use_warmup"]:
        sched = WarmupCosineScheduler(opt, warmup_steps=max(1, int(len(loader) * kw["warmup_frac"] * mc["base_epochs"])),
                                      total_steps=max(2, len(loader) * mc["base_epochs"]), base_lr=kw["base_lr"])
    base, base_hist = train_loop(base, loader, make_loader(Xv, yv, shuffle=False), opt, crit, max_epochs=mc["base_epochs"],
                                 patience=mc["base_patience"], verbose=False, scheduler=sched)
    out = {}
    for m, p in plan["modes"].items():
        Xft, yft = X[p["ft_train"]].astype(np.float32), y[p["ft_train"]]
        Xfv, yfv = X[p["ft_val"]].astype(np.float32), y[p["ft_val"]]
        Xev = X[p["eval"]].astype(np.float32)
        model = copy.deepcopy(base)
        for mod in model.modules():
            if isinstance(mod, nn.Dropout): mod.p = 0.65
        trainable = freeze_early_layers(model, keep_frac=kw["keep_frac"])
        crit_ft = nn.CrossEntropyLoss(weight=class_weights(yft, nclass).to(DEVICE), label_smoothing=kw["label_smoothing"])
        opt_ft = make_optimizer(trainable, kw["optimizer_name"], lr=kw["ft_lr"], weight_decay=kw["weight_decay"])
        ld = make_loader(Xft, yft)
        sch = None
        if kw["use_warmup"]:
            sch = WarmupCosineScheduler(opt_ft, warmup_steps=max(1, int(len(ld) * kw["warmup_frac"] * mc["ft_epochs"])),
                                        total_steps=max(2, len(ld) * mc["ft_epochs"]), base_lr=kw["ft_lr"])
        model, hist = train_loop(model, ld, make_loader(Xfv, yfv, shuffle=False), opt_ft, crit_ft, max_epochs=mc["ft_epochs"],
                                 patience=mc["ft_patience"], verbose=False, scheduler=sch)
        probs = predict_proba(model, Xev)
        assert probs.shape == (len(Xev), nclass) and np.isfinite(probs).all(), "bad probabilities"
        out[m] = dict(probs=probs, base_epochs=len(base_hist), ft_epochs=len(hist),
                      model=(copy.deepcopy(model).cpu().eval() if KEEP_MODELS else None))
        del model
    return out

## 7. Optional stage 0: cross-subject pretraining from other subjects' offline windows

Only runs if `EXT_SUBJECTS` is non-empty (each extra subject is a 16-24 GB download, processed into a small strided cache,
then the raw files are deleted). Only the **offline** session of those subjects is used and they can never be an evaluated subject.

In [ ]:
def build_finger_ext_pool(nclass):
    keep = CLASS_SETS[nclass]; Xs, ys, ts, off = [], [], [], 0
    for sid in EXT_SUBJECTS:
        zz = load_cache(cache_subject(sid, offline_only=True, stride=EXT_STRIDE))
        sel = np.isin(zz["y"], keep)
        Xs.append(zz["X"][sel].astype(np.float32)); ys.append(np.array([keep.index(int(l)) for l in zz["y"][sel]]))
        ts.append(zz["trial"][sel] + off); off += int(zz["trial"].max()) + 1
        print(f"  external subject S{sid:02d}: {int(sel.sum())} windows")
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(ts)

def pretrain_finger_ext(nclass, model_name, seed=0):
    mc = MODEL_CFG[model_name]; kw = mc["kw"]
    X, y, tid = build_finger_ext_pool(nclass)
    set_seed(seed)
    tr_m, va_m = split_by_trial(tid, y, 0.1, seed=0)
    assert not (set(tid[tr_m].tolist()) & set(tid[va_m].tolist())), "external train/validation share a trial"
    model = mc["factory"](N_CHANS, N_TIMES, nclass).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=class_weights(y[tr_m], nclass).to(DEVICE), label_smoothing=kw["label_smoothing"])
    opt = make_optimizer(model.parameters(), kw["optimizer_name"], lr=kw["base_lr"], weight_decay=kw["weight_decay"])
    loader = make_loader(X[tr_m], y[tr_m], batch_size=32)
    sched = None
    if kw["use_warmup"]:
        sched = WarmupCosineScheduler(opt, warmup_steps=max(1, int(len(loader) * 0.1 * EXT_EPOCHS)),
                                      total_steps=max(2, len(loader) * EXT_EPOCHS), base_lr=kw["base_lr"])
    model, hist = train_loop(model, loader, make_loader(X[va_m], y[va_m], batch_size=32, shuffle=False), opt, crit,
                             max_epochs=EXT_EPOCHS, patience=EXT_PATIENCE, verbose=True, scheduler=sched)
    print(f"  [{model_name}, {nclass}-class] external pretraining: {len(hist)} epochs, best validation accuracy {max(h['val_acc'] for h in hist):.3f}")
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

EXT_STATE = {}
if "cross" in VARIANTS:
    for k in FEASIBLE:
        for model_name in MODELS_TO_RUN:
            print(f"=== Stage 0: {k}-class, {model_name} ===")
            EXT_STATE[(k, model_name)] = pretrain_finger_ext(k, model_name)

## 8. Run: subject x class configuration x model x variant x seed (resumable)

In [ ]:
# EXPORT: ONNX helpers for the finger models (input 128 channels x 100 samples)
import zipfile
EXPORT_DIR = os.path.join(WORK_DIR, "finger_export"); os.makedirs(EXPORT_DIR, exist_ok=True)
try:
    import onnx, onnxruntime  # noqa
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"])
import onnxruntime as ort

FINGER_LABELS = {2: ["thumb", "pinky"], 3: ["thumb", "index", "pinky"]}   # order of CLASS_SETS: labels 1, (2), 4
MANIFEST_ENTRIES = []

def export_finger_model(model, k, mode, subj, mv, n_trials, per_window):
    mid = f"tcformer-finger{k}-s{subj:02d}-{mode.lower()}"
    path = os.path.join(EXPORT_DIR, mid + ".onnx")
    model = model.cpu().eval()
    dummy = torch.randn(4, N_CHANS, N_TIMES)
    kw = dict(input_names=["window"], output_names=["logits"], dynamic_axes={"window": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17)
    try:
        torch.onnx.export(model, dummy, path, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(model, dummy, path, **kw)
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    errs = []
    for n in (4, 37):
        x = torch.randn(n, N_CHANS, N_TIMES)
        with torch.no_grad():
            errs.append(float(np.abs(model(x).numpy() - sess.run(None, {"window": x.numpy()})[0]).max()))
    assert max(errs) < 1e-3, f"ONNX output differs from PyTorch by {max(errs):.2e} for {mid}; do not deploy this file"
    MANIFEST_ENTRIES.append({
        "id": mid, "task": f"finger_{k}", "mode": mode, "arch": "tcformer",
        "title": f"TCFormer, finger dataset subject S{subj:02d}, {k} fingers, {'executed' if mode == 'ME' else 'imagined'} movement",
        "label_order": FINGER_LABELS[k], "onnx_path": mid + ".onnx", "output": "logits",
        "input_shape": [N_CHANS, N_TIMES], "sampling_rate_hz": PARAMS["downsrate"],
        "trained_on": f"Finger dataset (KiltHub) subject S{subj:02d}, {mode} specialist, own earlier sessions only, seed 0, no cross-subject pretraining",
        "variant": "baseline",
        "protocol": "chronological, trial-level, session-held-out (pretrain on earlier sessions, fine-tune on first 30% of the last session, evaluate on the rest)",
        "warning": f"Personal model of subject S{subj:02d}: 128-channel BioSemi recordings resampled to 100 Hz; meaningless for other people or other EEG caps.",
        "evaluation": {"majority_vote_accuracy": float(mv), "n_heldout_trials": int(n_trials), "per_window_accuracy": float(per_window),
                       "chance": 1.0 / k, "seed": 0, "onnx_vs_torch_max_abs_diff": max(errs)}})
    print(f"  exported {mid}: majority vote {mv:.3f} on {n_trials} held-out trials (chance {1 / k:.2f}); ONNX parity {max(errs):.1e}")

def write_finger_replay(plan, m, probs, k, subj, mid):
    p = plan["modes"][m]; ev = p["eval"]
    tr, y = plan["trial"][ev], plan["y"][ev]
    order = np.argsort(tr, kind="stable"); labels = FINGER_LABELS[k]
    recs = [{"t": round(j * 0.125, 3), "trial": int(tr[i]), "true_class": labels[int(y[i])],
             "predicted_class": labels[int(probs[i].argmax())], "probs": {labels[c]: float(probs[i, c]) for c in range(k)}}
            for j, i in enumerate(order)]
    doc = {"label_order": labels, "channels_shown": [], "channel_names": [],
           "meta": {"source": f"Held-out trials of the exported model {mid} (not training data)", "model_id": mid, "subject": f"S{subj:02d}", "mode": m,
                    "n_windows": len(recs), "per_window_accuracy": float(np.mean([r["true_class"] == r["predicted_class"] for r in recs])),
                    "caveats": ["Single subject; no waveform is stored (128 channels).", "Time axis is replay time, not recording time."]},
           "records": recs}
    with open(os.path.join(EXPORT_DIR, f"replay_finger{k}_{m}.json"), "w") as fh:
        json.dump(doc, fh)


## 8. Run and export

In [ ]:
# EXPORT: train subject S01 once (TCFormer, seed 0) for 2 and 3 fingers, export every mode that can be evaluated
results = {}
t_start = time.time()
for subj in SUBJECTS:
    zd = load_cache(cache_subject(subj))
    for k in NCLASS_LIST:
        print(f"\n=== Subject {subj}, {k}-finger ===")
        plan = plan_finger(zd, k)
        n_eval = verify_finger_plan(plan, f"S{subj:02d} {k}-finger")
        print(f"  plan verified: held-out trials {n_eval}, test sessions {plan['test_sess']}, {len(plan['base_train'])} pretraining windows")
        t0 = time.time()
        out = run_finger_once(plan, "tcformer", "baseline", 0, None)
        print(f"  trained in {(time.time() - t0) / 60:.1f} min (check this against the time estimate)")
        for m, o in out.items():
            p = plan["modes"][m]; y_ev, tr_ev = plan["y"][p["eval"]], plan["trial"][p["eval"]]
            pred = o["probs"].argmax(1)
            mv, n = trial_vote_accuracy(pred, y_ev, tr_ev, k)
            pw = float((pred == y_ev).mean())
            results[f"S{subj:02d}|{k}finger|{m}"] = dict(majority_vote=mv, n_trials=n, per_window=pw, chance=1.0 / k)
            export_finger_model(o["model"], k, m, subj, mv, n, pw)
            write_finger_replay(plan, m, o["probs"], k, subj, MANIFEST_ENTRIES[-1]["id"])
        del plan, out
    del zd
print(f"\nTotal {(time.time() - t_start) / 60:.1f} min")
with open(RESULTS_PATH, "w") as fh:
    json.dump(results, fh, indent=1, default=_json_default)
if not MANIFEST_ENTRIES:
    print("NOTHING WAS EXPORTED. Read the messages above and tell Claude.")
else:
    with open(os.path.join(EXPORT_DIR, "manifest_entries.json"), "w") as fh:
        json.dump({"models": MANIFEST_ENTRIES}, fh, indent=1)
    zip_path = os.path.join(WORK_DIR, "neurobot_finger_models.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(os.listdir(EXPORT_DIR)):
            z.write(os.path.join(EXPORT_DIR, f), f)
    print(f"{len(MANIFEST_ENTRIES)} model(s) -> {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
    for e in MANIFEST_ENTRIES:
        ev = e["evaluation"]; print(" ", e["id"], "| majority vote", round(ev["majority_vote_accuracy"], 3), "on", ev["n_heldout_trials"], "trials | chance", round(ev["chance"], 2))


## After the run

1. Download **`neurobot_finger_models.zip`** from the Output tab of this version.
2. Send it back with the `majority vote ... on N held-out trials` lines.
3. The numbers will differ a little from Table IX because this run uses one seed and the table averages three.
